[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forestdatapartnership/whisp/blob/feat/viewers-timber-main/notebooks/crop_pathway_viewer.ipynb)

# Crop pathway viewer: decision tree linked to a map

Builds an interactive HTML page for the **annual-crop** (`risk_acrop`) or **perennial-crop** (`risk_pcrop`) decision tree: the tree on the left, a map on the right, linked. Click a question to see that indicator's pixels, click an outcome to see the pixels on that pathway, click again to clear. The combined outcome map shows low / more info needed / high for every pixel.

The tree is a display mirror of `risk.add_risk_acrop_col` / `add_risk_pcrop_col` (the same rule, drawn as a tree), and the map is built from the same indicator getters those functions use, so the per-pixel map and the per-plot table agree by construction.

**You need** a Google Earth Engine cloud project you can use. Set it in the first code cell. Everything runs on your account; nothing is sent anywhere else.

The page you build embeds map-tile links tied to your project that expire after about a day. Re-run the notebook to refresh them. The page itself can be opened or hosted anywhere for viewing; only the draw-a-plot analysis (last section) needs a local run.

WHISP is a non-authoritative exploration tool: outcomes are signals, not a legal determination.

In [ ]:
# >>> Set these <<<
PROJECT = "your-ee-project-id"   # your Earth Engine cloud project id, e.g. "ee-yourname"
COMMODITY = "acrop"              # "acrop" (annual crops) or "pcrop" (perennial crops)
NATIONAL = ["br"]                # national layers to include on top of the global ones (ISO2 codes); [] = global only
OUTPUT_HTML = f"{COMMODITY}_pathway_viewer.html"

## Install
The viewer modules live on the `feat/viewers-timber-main` branch of the whisp repository (they are not in the PyPI release yet). In Colab this installs that branch; on a local checkout of the branch it is a no-op. Restart the runtime after installing if `openforis_whisp` was already imported.

In [ ]:
import sys
try:
    from openforis_whisp import pathway_viewer  # noqa: F401
    print("viewer modules already importable, nothing to install")
except ImportError:
    !{sys.executable} -m pip install -q "git+https://github.com/forestdatapartnership/whisp.git@feat/viewers-timber-main"
    print("installed; if an older openforis_whisp was already imported, restart the runtime and run again")

In [ ]:
import ee
from IPython.display import HTML, display

from openforis_whisp import crop_map as cm
from openforis_whisp import decision_tree as dt
from openforis_whisp import pathway_viewer as pv
from openforis_whisp import pcrop_tree_export as pc
from openforis_whisp import acrop_tree_export as ac

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("Earth Engine ready:", PROJECT)

## The tree
One spec per commodity. It carries the tree, the pathway codes, the palette and the question-to-indicator mapping; the diagram, the map layers and the viewer are all generated from it.

In [ ]:
SPEC, RISK_USE_COL, RISK_COL, TITLE = {
    "acrop": (ac.ACROP_SPEC, "use_for_risk_acrop", "risk_acrop", "Annual-crop pathway viewer"),
    "pcrop": (pc.PCROP_SPEC, "use_for_risk_pcrop", "risk_pcrop", "Perennial-crop pathway viewer"),
}[COMMODITY]

print(dt.to_mermaid(SPEC))
print("pathway codes:", dt.code_names(SPEC))

## Build the map layers
One presence image per tree question (the union of the datasets the lookup table flags for that indicator and commodity), walked through the tree to a per-pixel pathway code, then collapsed to the three outcomes. Each layer becomes a signed tile URL for the map, plus one layer per tree node for the click-to-show behaviour.

In [ ]:
def s2_median(start, end):
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterDate(start, end)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
            .median())

land = ee.ImageCollection("ESA/WorldCover/v200").mosaic().gt(0)   # so 'no tree cover' shows on land, not sea
s2_vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
backgrounds = {
    "2020": cm.map_id_tile_url(s2_median("2020-01-01", "2020-12-31"), s2_vis),
    "2025": cm.map_id_tile_url(s2_median("2025-01-01", "2025-12-31"), s2_vis),
}

qimg = cm.build_crop_question_images(SPEC, RISK_USE_COL, national_codes=NATIONAL)
pathway = dt.eval_tree_ee(qimg, ee.Image, SPEC).updateMask(land)
outcome = cm.collapse_to_outcome3(SPEC, pathway)
palette = cm.crop_map_palette(SPEC)
tiles = {
    "outcome": cm.map_id_tile_url(outcome, palette["outcome_vis"]),
    "pathway": cm.map_id_tile_url(pathway.selfMask(), palette["pathway_vis"]),
}
# one layer per tree node: a question's presence pixels, or a pathway code's pixels
q_tiles = {q: cm.map_id_tile_url(img.selfMask().updateMask(land), {"min": 1, "max": 1, "palette": ["2b6cb0"]})
           for q, img in qimg.items()}
code_tiles = {c: cm.map_id_tile_url(pathway.eq(c).selfMask(), {"min": 1, "max": 1, "palette": [palette["code_colour"][c]]})
              for c in palette["code_names"]}
node_tiles = pv.node_tiles_by_id(SPEC, q_tiles, code_tiles, palette)
print("layers resolved:", len(tiles), "map layers,", len(node_tiles), "node layers")

## Assemble and show the viewer
Written to a single HTML file (in Colab: download it from the Files panel on the left, or run the download cell below). If the inline preview is blank, open the file in a browser; some notebook front ends strip scripts.

In [ ]:
BOOKMARKS = [
    ["Coffee (Minas Gerais)", -21.0, -46.0, 8],
    ["Cocoa (southern Bahia)", -15.0, -39.3, 9],
    ["Oil palm (Para)", -2.3, -48.6, 9],
    ["Soy frontier (Mato Grosso)", -12.5, -55.5, 8],
    ["Cocoa (Cote d Ivoire)", 6.7, -5.5, 9],
]

html = pv.build_pathway_viewer_html(
    SPEC, tiles, palette, TITLE,
    subtitle=f"{RISK_COL} tree + combined risk map",
    bookmarks=BOOKMARKS, backgrounds=backgrounds, risk_col=RISK_COL,
    center=(-12, -52), zoom=4, node_tiles=node_tiles,
)
with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)
print("wrote", OUTPUT_HTML)

import html as _h
display(HTML(f'<iframe srcdoc="{_h.escape(html)}" style="width:100%;height:820px;border:1px solid #ccc"></iframe>'))

In [ ]:
# In Colab: download the page to your machine (elsewhere this cell does nothing).
try:
    from google.colab import files  # noqa
    files.download(OUTPUT_HTML)
except ImportError:
    print("not in Colab; the file is at", OUTPUT_HTML)

## Optional, local only: draw a plot and analyse it
The page as built shows the map and the tree. To draw a plot and get its indicators and outcome from the whisp package (and move the threshold sliders), serve the page with the local proxy, which runs whisp on your Earth Engine project. In a terminal, from the folder holding the HTML:

```
python -m openforis_whisp.serve_viewer acrop_pathway_viewer.html 8788
```
then open http://127.0.0.1:8788/ in Chrome or Edge. This does not work from inside Colab.